# 06 — Deployment & API Consumption

> **Workflow position:** Model is trained, validated, and the business case is clear. Now we operationalise it so that the retention team can consume predictions — programmatically, via REST API, or in batch.

---

## Deployment Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                    CLIENT CHURN PREDICTION                  │
│                                                             │
│   Raw CSV  ──▶  Feature Engineering  ──▶  Pipeline (.joblib)│
│                                                    │        │
│              ┌─────────────────────────────────────┘        │
│              │                                              │
│              ▼                                              │
│   ┌──────────────────────┐    ┌──────────────────────────┐  │
│   │  Batch CLI           │    │  REST API (FastAPI)       │  │
│   │  python -m cli pred  │    │  POST /predict           │  │
│   │  → predictions.csv   │    │  → {prediction, score}   │  │
│   └──────────────────────┘    └──────────────────────────┘  │
│                                        │                    │
│                               ┌────────┘                    │
│                               ▼                             │
│                    CRM / Marketing Platform                 │
│                    (Salesforce / HubSpot / Brevo)           │
└─────────────────────────────────────────────────────────────┘
```

**Two consumption patterns:**

| Pattern | Use case | Latency | Volume |
|---|---|---|---|
| **Batch CLI** | Weekly risk scoring, campaign export | Minutes | Thousands of rows |
| **REST API** | Real-time in-app scoring, CRM integration | <100ms | One record at a time |

## Setup

In [ ]:
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid")
PALETTE = sns.color_palette("Set2")
sns.set_palette(PALETTE)

print("Setup complete. Project root:", PROJECT_ROOT)

## Ensure Model is Trained

Before generating predictions, the model artefact must exist. This cell loads an existing model or trains a fresh one if needed.

In [ ]:
import joblib
from client_churn_prediction.config import load_config
from client_churn_prediction.models import train

config = load_config()
model_path = PROJECT_ROOT / "models" / "model.joblib"

if model_path.exists():
    pipeline = joblib.load(model_path)
    print(f"Loaded existing model from: {model_path}")
    print(f"Model type: {type(pipeline.named_steps.get('model', pipeline)).__name__}")
else:
    print("No model found. Training now...")
    result = train()
    pipeline = joblib.load(model_path)
    print(f"Model trained and saved to: {result['model_path']}")

# Load and display last known metrics
metrics_path = PROJECT_ROOT / "reports" / "metrics.json"
if metrics_path.exists():
    with open(metrics_path) as f:
        metrics = json.load(f)
    print(f"\nModel validation metrics:")
    print(f"  ROC AUC           : {metrics.get('roc_auc', 'N/A')}")
    print(f"  Average Precision : {metrics.get('average_precision', 'N/A')}")
    print(f"  F1 Score          : {metrics.get('f1', 'N/A')}")
    print(f"  Train size        : {metrics.get('train_size', 'N/A')} rows")
    print(f"  Validation size   : {metrics.get('valid_size', 'N/A')} rows")

## Batch Prediction Demo

The `predict()` function from our `models` module:
1. Loads the serialised pipeline from `models/model.joblib`
2. Reads an input CSV file
3. Applies the same feature engineering used during training (preventing leakage)
4. Outputs predictions with a `churn_score` column (0-1 probability) and a binary `prediction` column
5. Saves the enriched DataFrame to `data/processed/predictions.csv`

### Preparing a sample batch

In [ ]:
from client_churn_prediction.data import load_training_frame

# Use a slice of the dataset as a "new batch" (simulating production)
df_all = load_training_frame(config)

# Take 20 random records as our prediction batch (drop the target column)
target_col = config.get("data", {}).get("target", "Exited")
batch = df_all.sample(20, random_state=99).copy()
true_labels = batch[target_col].values if target_col in batch.columns else None

# Save batch WITHOUT the target column
batch_path = PROJECT_ROOT / "data" / "processed" / "batch_input.csv"
batch_path.parent.mkdir(parents=True, exist_ok=True)
batch_no_target = batch.drop(columns=[target_col], errors="ignore")
batch_no_target.to_csv(batch_path, index=False)

print(f"Batch saved: {batch_path}")
print(f"Shape: {batch_no_target.shape}")
batch_no_target.head(3)

### Run batch prediction

In [ ]:
from client_churn_prediction.models import predict

output_path = PROJECT_ROOT / "data" / "processed" / "predictions.csv"
saved_to = predict(
    input_path=str(batch_path),
    output_path=str(output_path)
)

print(f"Predictions written to: {saved_to}")

### Display sample predictions with churn_score

In [ ]:
preds_df = pd.read_csv(output_path)

# Attach true labels for evaluation
if true_labels is not None:
    preds_df["true_label"] = true_labels

# Sort by churn risk — highest risk first (as a campaign list would be sorted)
display_cols = [c for c in ["customerid", "age", "geography", "balance", "num_of_products",
                             "is_active_member", "churn_score", "prediction", "true_label"]
                if c in preds_df.columns]
preds_sorted = preds_df[display_cols].sort_values("churn_score", ascending=False)

print("=" * 70)
print("TOP CHURN RISK PREDICTIONS (sorted by churn_score, highest first)")
print("=" * 70)
print(preds_sorted.to_string(index=False, float_format="{:.4f}".format))

# Summary statistics
print(f"\nScore distribution:")
print(preds_df["churn_score"].describe().round(3))
print(f"\nPredicted churners: {preds_df['prediction'].sum()} / {len(preds_df)}")
if true_labels is not None:
    print(f"Actual churners   : {sum(true_labels)} / {len(true_labels)}")

In [ ]:
# Visualise churn score distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.hist(preds_df["churn_score"], bins=20, color=PALETTE[0], edgecolor="white", alpha=0.85)
ax.axvline(0.5, color="crimson", linestyle="--", lw=1.5, label="Default threshold (0.5)")
ax.set_title("Distribution of Churn Scores (Batch)", fontsize=12)
ax.set_xlabel("Churn Probability Score", fontsize=11)
ax.set_ylabel("Count", fontsize=11)
ax.legend(fontsize=10)

ax2 = axes[1]
colors_pred = [PALETTE[3] if p == 1 else PALETTE[2] for p in preds_df["prediction"]]
ax2.scatter(
    range(len(preds_sorted)), preds_sorted["churn_score"],
    c=[PALETTE[3] if p == 1 else PALETTE[2] for p in preds_sorted["prediction"]],
    s=80, alpha=0.85, edgecolors="white"
)
ax2.axhline(0.5, color="grey", linestyle="--", lw=1.2, label="Threshold 0.50")
ax2.set_title("Churn Score by Client (Ranked)", fontsize=12)
ax2.set_xlabel("Client rank (highest risk first)", fontsize=11)
ax2.set_ylabel("Churn Score", fontsize=11)

# Legend proxies
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=PALETTE[3], label="Predicted: Churn"),
    Patch(facecolor=PALETTE[2], label="Predicted: Stay"),
]
ax2.legend(handles=legend_elements, fontsize=10)

sns.despine()
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / "06_prediction_scores.png", dpi=150, bbox_inches="tight")
plt.show()

## FastAPI Structure Walkthrough

The REST API is defined in `src/client_churn_prediction/api.py`. Here is the full source code:

In [ ]:
api_path = PROJECT_ROOT / "src" / "client_churn_prediction" / "api.py"
print("="*60)
print("FILE: src/client_churn_prediction/api.py")
print("="*60)
print(api_path.read_text(encoding="utf-8"))

### API Design Highlights

| Component | Details |
|---|---|
| **Framework** | FastAPI — automatic OpenAPI docs at `/docs`, async-ready, Pydantic validation |
| **Input schema** | `RecordsPayload` — list of dicts, same columns as training data |
| **Output schema** | `{"prediction": [0,1,...], "score": [0.12, 0.87,...]}` |
| **Health check** | `GET /health` → `{"status": "ok"}` for load-balancer probes |
| **Model loading** | Model loaded on each request from `models/model.joblib` (can be optimised with a startup event for production) |
| **Column normalisation** | `normalize_columns()` ensures `CreditScore` and `credit_score` are treated identically |

**OpenAPI docs** are automatically available at `http://localhost:8000/docs` once the server is running.

## Sample API Request

The cell below shows exactly how a downstream system (CRM, mobile app, web service) would call the API. The `requests.post()` call is commented out so this notebook runs without a running server, but the expected response is printed.

In [ ]:
import json

# --- Sample payload: one customer record ---
# Field names are case-insensitive — the API normalises them internally.
sample_payload = {
    "records": [
        {
            "CreditScore": 619,
            "Geography": "France",
            "Gender": "Female",
            "Age": 42,
            "Tenure": 2,
            "Balance": 0.0,
            "NumOfProducts": 1,
            "HasCrCard": 1,
            "IsActiveMember": 1,
            "EstimatedSalary": 101349
        }
    ]
}

print("Request payload:")
print(json.dumps(sample_payload, indent=2))

# --- Uncomment to make a live request ---
# import requests
# response = requests.post("http://localhost:8000/predict", json=sample_payload)
# print("\nAPI Response:")
# print(json.dumps(response.json(), indent=2))

print("\nExpected API response:")
print(json.dumps({"prediction": [1], "score": [0.73]}, indent=2))

In [ ]:
# --- Simulate the API call directly (no server required) ---
# This replicates exactly what the API endpoint does internally.
import pandas as pd
from client_churn_prediction.data import normalize_columns
from client_churn_prediction.features import model_matrix, prepare_features

df_request = normalize_columns(pd.DataFrame(sample_payload["records"]))
X_request, _, _ = model_matrix(prepare_features(df_request, config, training=False), config, training=False)

prediction = pipeline.predict(X_request).tolist()
score = pipeline.predict_proba(X_request)[:, 1].tolist()

response_simulated = {"prediction": prediction, "score": [round(s, 4) for s in score]}
print("Simulated API response (no server needed):")
print(json.dumps(response_simulated, indent=2))
print()
print(f"Churn probability: {score[0]:.1%}")
print(f"Verdict          : {'HIGH RISK — flag for retention campaign' if prediction[0] == 1 else 'LOW RISK — no action needed'}")

## Batch API Requests

The API accepts multiple records in a single request for efficient batch processing:

In [ ]:
# Simulate a batch API call with 5 diverse customer profiles
batch_payload = {
    "records": [
        {"CreditScore": 619, "Geography": "France",  "Gender": "Female", "Age": 42, "Tenure": 2,  "Balance": 0.0,      "NumOfProducts": 1, "HasCrCard": 1, "IsActiveMember": 1, "EstimatedSalary": 101349},
        {"CreditScore": 850, "Geography": "Spain",   "Gender": "Male",   "Age": 35, "Tenure": 8,  "Balance": 125000.0,  "NumOfProducts": 2, "HasCrCard": 1, "IsActiveMember": 1, "EstimatedSalary": 149748},
        {"CreditScore": 450, "Geography": "Germany", "Gender": "Female", "Age": 58, "Tenure": 1,  "Balance": 0.0,      "NumOfProducts": 1, "HasCrCard": 0, "IsActiveMember": 0, "EstimatedSalary": 45000},
        {"CreditScore": 720, "Geography": "France",  "Gender": "Male",   "Age": 29, "Tenure": 5,  "Balance": 78000.0,  "NumOfProducts": 2, "HasCrCard": 1, "IsActiveMember": 1, "EstimatedSalary": 92000},
        {"CreditScore": 550, "Geography": "Germany", "Gender": "Female", "Age": 65, "Tenure": 10, "Balance": 200000.0, "NumOfProducts": 1, "HasCrCard": 1, "IsActiveMember": 0, "EstimatedSalary": 35000},
    ]
}

df_batch = normalize_columns(pd.DataFrame(batch_payload["records"]))
X_batch, _, _ = model_matrix(prepare_features(df_batch, config, training=False), config, training=False)

batch_preds  = pipeline.predict(X_batch).tolist()
batch_scores = pipeline.predict_proba(X_batch)[:, 1].tolist()

result_df = df_batch[[ "geography", "gender", "age", "num_of_products", "is_active_member", "balance"]].copy()
result_df["churn_score"] = [round(s, 4) for s in batch_scores]
result_df["prediction"]  = batch_preds
result_df["risk_level"]  = result_df["churn_score"].apply(
    lambda s: "HIGH" if s > 0.7 else ("MEDIUM" if s > 0.4 else "LOW")
)
result_df = result_df.sort_values("churn_score", ascending=False)

print("Batch API results (5 customers):")
print(result_df.to_string(index=False))

## How to Run the API Locally

### Prerequisites

```bash
# 1. Clone the repo
git clone https://github.com/your-username/client-churn-prediction.git
cd client-churn-prediction

# 2. Create a virtual environment (Python 3.10+)
python -m venv .venv
source .venv/bin/activate  # On Windows: .venv\Scripts\activate

# 3. Install dependencies
pip install -r requirements.txt
pip install -r requirements-api.txt
```

### Train the model

```bash
# Option A: via Python
PYTHONPATH=src python -c "from client_churn_prediction.models import train; train()"

# Option B: via Makefile (if available)
make train
```

### Start the API server

```bash
PYTHONPATH=src uvicorn client_churn_prediction.api:app --reload --port 8000
```

The server starts at **http://localhost:8000**

### Explore the interactive docs

Open **http://localhost:8000/docs** in your browser — FastAPI auto-generates an interactive Swagger UI where you can:
- Browse all endpoints
- Try requests live from the browser
- Download the OpenAPI schema

### Quick test with curl

```bash
curl -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"records": [{"CreditScore": 619, "Geography": "France", "Gender": "Female", "Age": 42, "Tenure": 2, "Balance": 0, "NumOfProducts": 1, "HasCrCard": 1, "IsActiveMember": 1, "EstimatedSalary": 101349}]}'
```

In [ ]:
# Print startup instructions inline
procfile_path = PROJECT_ROOT / "Procfile"
if procfile_path.exists():
    print("Procfile content (used by hosting platforms):")
    print("  ", procfile_path.read_text().strip())

print()
print("To start the API server manually:")
print("  PYTHONPATH=src uvicorn client_churn_prediction.api:app --reload --port 8000")
print()
print("Health check:")
print("  curl http://localhost:8000/health")
print()
print("Swagger UI:")
print("  http://localhost:8000/docs")

## Cloud Deployment Notes

The project ships with a `Procfile` (`web: PYTHONPATH=src uvicorn client_churn_prediction.api:app --host 0.0.0.0 --port $PORT`) that makes it compatible with all major PaaS providers.

### Option A — Render (recommended for free tier)

1. Push the repo to GitHub
2. Create a new **Web Service** on [render.com](https://render.com)
3. Connect the GitHub repo
4. Set build command: `pip install -r requirements.txt && pip install -r requirements-api.txt`
5. Set start command: `PYTHONPATH=src uvicorn client_churn_prediction.api:app --host 0.0.0.0 --port $PORT`
6. Add env var: `PYTHONPATH=src`

**Important:** Train the model and commit `models/model.joblib` to the repo before deploying (or add a training step to the build command).

### Option B — Railway

Railway auto-detects the `Procfile`:
```bash
railway login
railway new
railway up
```

### Option C — Heroku

```bash
heroku create your-churn-api
heroku buildpacks:set heroku/python
git push heroku main
```

Heroku reads the `Procfile` automatically.

### Environment variables for cloud

| Variable | Purpose |
|---|---|
| `PYTHONPATH` | Must include `src` so modules are importable |
| `PORT` | Set automatically by the platform — do not hard-code |
| `MODEL_PATH` | Override default path if storing model in object storage |

## Google Colab Quick-Start

Recruiters and reviewers can run the full project in a free browser-based environment without installing anything locally.

### Step-by-step for Google Colab

Copy and run this cell in a **new Colab notebook**:

```python
# ============================================================
# CLIENT CHURN PREDICTION — Google Colab Quick-Start
# Run this cell first to set up the entire project
# ============================================================

# 1. Clone the repository
!git clone https://github.com/your-username/client-churn-prediction.git
%cd client-churn-prediction

# 2. Install dependencies
!pip install -q -r requirements.txt

# 3. Set Python path so imports work
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 4. Download the dataset from Kaggle (requires Kaggle API credentials)
# Option A: If you have Kaggle credentials
# !pip install -q kaggle
# !kaggle datasets download -d shrutimechlearn/churn-modelling --unzip -p data/raw/

# Option B: Manual upload to Colab
# from google.colab import files
# uploaded = files.upload()  # Upload Churn_Modelling.csv
# import shutil
# shutil.move("Churn_Modelling.csv", "data/raw/Churn_Modelling.csv")

# 5. Train the model
from client_churn_prediction.models import train
result = train()
print("Model trained! Metrics:", result["metrics"])

# 6. Open the notebooks
# Run: jupyter notebook (or use Colab's built-in .ipynb runner)
```

### Alternative: Upload dataset directly in Colab

The Kaggle dataset ([Churn Modelling](https://www.kaggle.com/datasets/shrutimechlearn/churn-modelling)) is free to download after logging in. Download `Churn_Modelling.csv` and upload it to `data/raw/` in the Colab file browser.

### Colab resource tips

- **Free T4 GPU**: not needed for this project (CPU is sufficient)
- **RAM**: default Colab RAM (12 GB) is more than enough for 10K rows
- **Persistence**: Colab sessions expire after ~12 hours; model artefacts are lost. Re-run the training cell when resuming.

In [ ]:
# Final project summary
print("=" * 65)
print("PROJECT SUMMARY — Client Churn Prediction")
print("=" * 65)

summary = {
    "Dataset": "Kaggle Churn Modelling (10,000 rows, 14 features)",
    "Target": "Exited (binary: 1=churned, 0=stayed)",
    "Churn rate": "~20% (imbalanced)",
    "Model": "GradientBoostingClassifier (sklearn)",
    "Preprocessing": "ColumnTransformer (median + StandardScaler + OneHotEncoder)",
    "Feature engineering": "8 domain features (balance ratios, engagement signals, age groups)",
    "Validation strategy": "Stratified 80/20 split + 5-fold CV comparison",
    "Primary metric": "ROC AUC",
    "Deployment": "FastAPI REST API (POST /predict) + batch CLI",
    "Key notebooks": "04 model comparison, 05 tuning + ROI, 06 deployment",
}

for k, v in summary.items():
    print(f"  {k:<25}: {v}")

print()
if metrics_path.exists():
    with open(metrics_path) as f:
        m = json.load(f)
    print("Final model performance:")
    for metric in ["roc_auc", "average_precision", "f1", "precision", "recall"]:
        if metric in m:
            print(f"  {metric:<25}: {m[metric]:.4f}")

print()
print("API endpoints:")
print("  GET  /health   → {status: ok}")
print("  POST /predict  → {prediction: [...], score: [...]}")
print("  GET  /docs     → Swagger UI")